# 작업 시작 전 위험도 시험 평가

CSV의 시점·제외조건 설정을 읽는 사전평가입니다. 위에서 아래로 모두 실행하세요. 기존 01·02 노트북과 원본 CSV는 변경하지 않습니다.

시점/가연물-거리 분기는 구현됐지만 법령 자동경고, 모든 해당없음 응답의 타당성, MSDS 파생변수는 미구현입니다. 등급은 잠정값이며 최종 안전판정은 보류합니다.

설정은 data/09_사전평가설정.csv에서 관리합니다. 기존 엑셀을 CSV로 재변환해도 이 추가 설정 파일을 삭제하지 마세요.


In [1]:
import json
from pathlib import Path
from collections.abc import Mapping
import math
import pandas as pd
from pandas.api.types import is_scalar, is_bool

# 실행 위치가 프로젝트 루트 또는 notebooks 폴더여도 data 폴더를 찾습니다.
def find_data_dir():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [parent / "data", parent / "fire_risk_project" / "data"]:
            if (candidate / "08_점수규칙DB.csv").is_file():
                return candidate
    raise FileNotFoundError("프로젝트 data 폴더를 찾을 수 없습니다.")

data_dir = find_data_dir()

def read_database(filename):
    table = pd.read_csv(
        data_dir / filename, encoding="utf-8-sig",
        dtype=str, keep_default_na=False,
    )
    # ID와 선택지 양끝의 불필요한 공백만 정리합니다.
    return table.apply(lambda column: column.str.strip())

tasks = read_database("01_공정_세부작업DB.csv")
questions = read_database("02_위험요인_입력항목DB.csv")
score_rules = read_database("08_점수규칙DB.csv")
standards = read_database("04_법령_KOSHA기준DB.csv")

print("읽은 폴더:", data_dir)
print(f"작업 {len(tasks)}개 / 질문 {len(questions)}개 / 규칙 조건 {len(score_rules)}개")

prework_policies = read_database("09_사전평가설정.csv")


읽은 폴더: c:\Users\uijun\OneDrive\바탕 화면\Project\fire_risk_project\data
작업 62개 / 질문 41개 / 규칙 조건 138개


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\uijun\\OneDrive\\바탕 화면\\Project\\fire_risk_project\\data\\09_사전평가설정.csv'

## 1. 공통 함수

- `get_question()`: 질문 조회를 한곳에서 처리합니다.
- `to_number()`: 빈칸·무한대·참/거짓 등을 숫자로 잘못 계산하지 않도록 합니다.
- `matches_numeric_rule()`: CSV에 저장된 비교연산자와 구간 경계를 해석합니다.
- `build_score_result()`: 선택형과 숫자형이 같은 결과 형식을 사용하게 합니다.

점수·가중치·정규화 기준·등급 경계는 CSV에서 읽습니다. CSV의 규칙 문장을 임의로 실행하지 않습니다.


In [ ]:
def split_ids(value):
    return [item.strip() for item in value.split(";") if item.strip()]

def is_missing(value):
    if not is_scalar(value):
        raise ValueError("응답은 목록이나 사전이 아닌 단일 값이어야 합니다.")
    return bool(pd.isna(value)) or (isinstance(value, str) and not value.strip())

def to_number(value, label):
    if is_bool(value) or is_missing(value):
        raise ValueError(f"{label}: 숫자가 필요합니다.")
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        raise ValueError(f"{label}: 숫자로 변환할 수 없습니다.") from None
    if not math.isfinite(number):
        raise ValueError(f"{label}: 유한한 숫자가 필요합니다.")
    return number

def read_flag(value, label):
    number = to_number(value, label)
    if number not in (0, 1):
        raise ValueError(f"{label}: 0 또는 1이어야 합니다.")
    return number == 1

def get_single_row(table, label):
    if len(table) != 1:
        raise ValueError(f"{label}: 해당 행이 없거나 중복됩니다.")
    return table.iloc[0]

def get_question(question_id):
    question_id = str(question_id).strip()
    return get_single_row(
        questions.loc[questions["항목ID"] == question_id],
        f"질문 {question_id}",
    )

def get_rules(rule_id):
    result = score_rules.loc[score_rules["점수규칙ID"] == rule_id]
    if result.empty:
        raise ValueError(f"존재하지 않는 점수규칙: {rule_id}")
    return result

def matches_numeric_rule(number, rule):
    operator = rule["비교연산자"]
    lower = to_number(rule["기준값1"], "기준값1")

    if operator == "<":
        return number < lower
    if operator == "<=":
        return number <= lower
    if operator == ">":
        return number > lower
    if operator == ">=":
        return number >= lower
    if operator == "EQ":
        return number == lower
    if operator == "구간":
        upper = to_number(rule["기준값2"], "기준값2")
        if lower > upper:
            raise ValueError("구간의 하한이 상한보다 큽니다.")
        lower_included = read_flag(rule["하한포함"], "하한포함")
        upper_included = read_flag(rule["상한포함"], "상한포함")
        lower_ok = number >= lower if lower_included else number > lower
        upper_ok = number <= upper if upper_included else number < upper
        return lower_ok and upper_ok
    raise NotImplementedError(f"지원하지 않는 비교연산자: {operator}")

def build_score_result(question, answer, rule):
    score = to_number(rule["점수"], "점수")
    weight = to_number(rule["가중치"], "가중치")
    maximum = to_number(rule["정규화최대점수"], "정규화최대점수")
    if maximum <= 0 or weight < 0 or not 0 <= score <= maximum:
        raise ValueError("점수·가중치·정규화 기준의 범위를 확인하세요.")

    included = weight > 0
    evidence = list(dict.fromkeys(
        split_ids(question["연결기준ID"]) + split_ids(rule["근거ID"])
    ))
    return {
        "항목ID": question["항목ID"],
        "확인항목": question["확인항목"],
        "사용자응답": answer,
        "단위": question["단위"],
        "점수규칙ID": rule["점수규칙ID"],
        "조건순번": rule["조건순번"],
        "점수": score,
        "가중치": weight,
        "가중점수": score * weight if included else 0.0,
        "최대가중점수": maximum * weight if included else 0.0,
        "점수합산대상": included,
        "중복집계그룹": question["중복집계그룹"],
        "그룹집계방식": question["그룹집계방식"],
        "관련기준ID": ";".join(evidence),
        "적용조건": rule["적용조건"],
        "확정상태": rule["확정상태"],
    }


## 2. 원본 DB 연결 확인

원본 H05 질문은 여전히 9개입니다. 아래 사전평가 분기에서 사후 질문을 제외합니다.


In [ ]:
def validate_databases():
    requirements = [
        (tasks, ["작업ID"]),
        (questions, ["항목ID", "적용작업ID", "확인항목", "입력형식",
                     "선택지·허용값", "단위", "필수여부", "점수규칙ID",
                     "연결기준ID", "중복집계그룹", "그룹집계방식"]),
        (score_rules, ["점수규칙ID", "조건순번", "규칙유형", "입력값·조건",
                       "비교연산자", "기준값1", "기준값2", "점수", "가중치",
                       "정규화최대점수", "하한포함", "상한포함",
                       "근거ID", "적용조건", "확정상태"]),
        (standards, ["기준ID"]),
    ]
    for table, columns in requirements:
        missing = set(columns) - set(table.columns)
        if missing:
            raise ValueError(f"필수 컬럼 누락: {sorted(missing)}")
    for table, key in [(tasks, "작업ID"), (questions, "항목ID"), (standards, "기준ID")]:
        if table[key].eq("").any() or table[key].duplicated().any():
            raise ValueError(f"{key}: 빈값 또는 중복")

    sequences = score_rules["조건순번"].map(lambda x: to_number(x, "조건순번"))
    if not sequences.map(lambda x: x >= 1 and x.is_integer()).all():
        raise ValueError("조건순번은 1 이상의 정수여야 합니다.")
    rule_keys = list(zip(score_rules["점수규칙ID"], sequences))
    if score_rules["점수규칙ID"].eq("").any() or len(set(rule_keys)) != len(rule_keys):
        raise ValueError("규칙ID가 비었거나 규칙ID+조건순번이 중복됩니다.")

    task_ids = set(tasks["작업ID"])
    rule_ids = set(score_rules["점수규칙ID"])
    standard_ids = set(standards["기준ID"])
    for _, question in questions.iterrows():
        if not split_ids(question["적용작업ID"]):
            raise ValueError(f"{question['항목ID']}: 적용작업ID 누락")
        if set(split_ids(question["적용작업ID"])) - task_ids:
            raise ValueError(f"{question['항목ID']}: 연결된 작업ID 오류")
        if question["점수규칙ID"] not in rule_ids:
            raise ValueError(f"{question['항목ID']}: 연결된 점수규칙ID 오류")
        if set(split_ids(question["연결기준ID"])) - standard_ids:
            raise ValueError(f"{question['항목ID']}: 연결된 기준ID 오류")
        read_flag(question["필수여부"], "필수여부")
        if not question["중복집계그룹"] or question["그룹집계방식"] != "MAX_PER_TASK":
            raise ValueError(f"{question['항목ID']}: 중복집계 설정 오류")

    for _, rule in score_rules.iterrows():
        if set(split_ids(rule["근거ID"])) - standard_ids:
            raise ValueError(f"{rule['점수규칙ID']}: 근거ID 오류")
    print("DB의 기본 연결·중복·집계 설정 검사 통과")

def get_questions_for_task(task_id):
    task_id = str(task_id).strip()
    get_single_row(tasks.loc[tasks["작업ID"] == task_id], f"작업 {task_id}")
    applies = questions["적용작업ID"].map(lambda value: task_id in split_ids(value))
    return questions.loc[applies].copy().reset_index(drop=True)

validate_databases()
h05_questions = get_questions_for_task("H05")
print("H05 질문 수:", len(h05_questions))
display(h05_questions[["항목ID", "확인항목", "입력형식", "점수규칙ID"]])


DB의 기본 연결·중복·집계 설정 검사 통과
H05 질문 수: 9


,항목ID,확인항목,입력형식,점수규칙ID
0,Q001,주변 가연물 존재 여부,선택,S001
1,Q002,가연물과 작업점 거리,숫자,S002
2,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,선택,S003
3,Q004,화재감시자 배치 여부,선택,S004
4,Q018,용접방화포 사용 시 성능인증 제품 여부,선택,S019
5,Q019,작업점 하부 가연물의 불티 착화 우려 여부,선택,S020
6,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,선택,S021
7,Q025,초기소화설비 비치·접근 가능 여부,선택,S026
8,Q026,작업 종료 후 잔불·불티 확인 여부,선택,S027


## 3. 응답 점수화

이제 `score_answer()` 하나가 입력형식을 확인해 선택형 또는 숫자형 규칙을 찾습니다. 앞서 만든 별도의 `score_numeric_answer()`를 호출할 필요가 없습니다.

가스농도 Q030처럼 원시 입력을 파생변수로 변환해야 하는 질문은 아직 처리하지 않으며, 명시적인 오류로 알려줍니다.


In [ ]:
def score_answer(question_id, answer):
    question = get_question(question_id)
    if is_missing(answer):
        raise ValueError(f"{question['항목ID']}: 응답이 비었습니다.")
    rules = get_rules(question["점수규칙ID"])

    if question["입력형식"] == "선택":
        answer = str(answer).strip()
        allowed = [value.strip() for value in question["선택지·허용값"].split("/")]
        if answer not in allowed:
            raise ValueError(f"{question['항목ID']} 허용 응답: {allowed}")
        if not ((rules["규칙유형"] == "범주형") & (rules["비교연산자"] == "EQ")).all():
            raise ValueError("선택형 질문의 규칙유형·연산자를 확인하세요.")
        matched = rules.loc[rules["입력값·조건"] == answer]

    elif question["입력형식"] == "숫자":
        if "/" in question["단위"] or not rules["적용대상"].eq("02_위험요인").all():
            raise NotImplementedError("단위 환산·MSDS 파생변수 계산이 먼저 필요합니다.")
        answer = to_number(answer, question["확인항목"])
        if question["선택지·허용값"] != "0 이상":
            raise NotImplementedError("지원하지 않는 숫자 입력 제한입니다.")
        if answer < 0:
            raise ValueError("0 이상의 숫자를 입력하세요.")
        if not rules["규칙유형"].isin(["수치형", "범위형"]).all():
            raise ValueError("숫자형 질문의 규칙유형을 확인하세요.")
        mask = rules.apply(lambda rule: matches_numeric_rule(answer, rule), axis=1)
        matched = rules.loc[mask]
    else:
        raise NotImplementedError(f"지원하지 않는 입력형식: {question['입력형식']}")

    rule = get_single_row(matched, f"{question['점수규칙ID']} 응답 {answer}")
    return build_score_result(question, answer, rule)

display(pd.DataFrame([
    score_answer("Q004", "미배치"),
    score_answer("Q004", "해당없음"),
    score_answer("Q002", 10.9),
    score_answer("Q002", 11),
]))


,항목ID,확인항목,사용자응답,단위,점수규칙ID,조건순번,점수,가중치,가중점수,최대가중점수,점수합산대상,중복집계그룹,그룹집계방식,관련기준ID,적용조건,확정상태
0,Q004,화재감시자 배치 여부,미배치,-,S004,1,3.0,2.0,6.0,6.0,True,Q004,MAX_PER_TASK,R002,법령상 배치대상 조건 충족 시 적용,미확정
1,Q004,화재감시자 배치 여부,해당없음,-,S004,3,0.0,0.0,0.0,0.0,False,Q004,MAX_PER_TASK,R002,법령상 배치대상 조건 충족 시 적용,미확정
2,Q002,가연물과 작업점 거리,10.9,m,S002,1,1.0,1.5,1.5,4.5,True,Q002,MAX_PER_TASK,R001;R002,화기·불티 작업,미확정
3,Q002,가연물과 작업점 거리,11.0,m,S002,2,0.0,1.5,0.0,4.5,True,Q002,MAX_PER_TASK,R001;R002,화기·불티 작업,미확정


## 4. 사전 질문 분기와 합산

사후 질문은 시점제외, 가연물이 없을 때 거리 질문은 조건제외로 표시합니다. 선행응답이 빠지면 대기로 남기고 전체 계산을 막습니다. 제외 질문에 예전 응답이 남아도 분자·분모에 포함하지 않습니다. 제외한 질문의 점수는 빈값이며 제외 사유와 기존 입력은 로그에 남습니다.

Q041은 도장 H19에서는 제외하지만, 후속 열풍 건조 H21에서는 착수 전 확인으로 유지합니다. 작업ID별 구체 설정이 기본 설정보다 우선합니다.


In [ ]:
def normalize_answers(answers):
    if not isinstance(answers, Mapping):
        raise ValueError("응답은 {질문ID: 응답} 형식의 사전이어야 합니다.")
    normalized = {}
    for key, value in answers.items():
        key = str(key).strip()
        if not key or key in normalized:
            raise ValueError(f"빈 질문ID 또는 공백 정리 후 중복된 질문ID: {key}")
        if not is_scalar(value):
            raise ValueError(f"{key}: 단일 응답값이 필요합니다.")
        normalized[key] = value
    return normalized

def get_prework_policy(question_id, task_id):
    policies = prework_policies.loc[prework_policies["항목ID"] == question_id]
    specific = policies.loc[
        policies["적용작업ID"].map(lambda value: task_id in split_ids(value))
    ]
    if not specific.empty:
        return get_single_row(specific, f"{task_id}/{question_id} 사전평가 설정")
    return get_single_row(
        policies.loc[policies["적용작업ID"] == "*"],
        f"{question_id} 기본 사전평가 설정",
    )

def validate_prework_policies():
    required_columns = [
        "항목ID", "적용작업ID", "사전평가사용", "사전확인항목",
        "사전안내", "제외조건_JSON", "설정상태",
    ]
    if set(required_columns) - set(prework_policies.columns):
        raise ValueError("사전평가 설정 CSV의 필수 컬럼이 없습니다.")
    question_ids = set(questions["항목ID"])
    if set(prework_policies["항목ID"]) != question_ids:
        raise ValueError("사전평가 설정과 질문ID 목록이 일치하지 않습니다.")
    for _, policy in prework_policies.iterrows():
        read_flag(policy["사전평가사용"], "사전평가사용")
        question = get_question(policy["항목ID"])
        if policy["적용작업ID"] != "*":
            if not split_ids(policy["적용작업ID"]) or (
                set(split_ids(policy["적용작업ID"])) - set(split_ids(question["적용작업ID"]))
            ):
                raise ValueError("사전평가 설정의 적용작업ID를 확인하세요.")
        condition_text = policy["제외조건_JSON"]
        if condition_text:
            condition = json.loads(condition_text)
            if (
                not isinstance(condition, dict)
                or set(condition) != {"rule_id", "operator", "value"}
                or condition["operator"] != "ALL_EQ"
            ):
                raise ValueError("지원하지 않는 질문 제외조건입니다.")
            parent_rules = get_rules(condition["rule_id"])
            if not parent_rules["규칙유형"].eq("범주형").all():
                raise ValueError("선행응답 조건은 범주형 규칙만 지원합니다.")
            if condition["value"] not in set(parent_rules["입력값·조건"]):
                raise ValueError("제외조건의 응답값이 선행 규칙에 없습니다.")
            if condition["rule_id"] == question["점수규칙ID"]:
                raise ValueError("질문이 자기 응답으로 제외 여부를 결정할 수 없습니다.")
    for _, question in questions.iterrows():
        for task_id in split_ids(question["적용작업ID"]):
            get_prework_policy(question["항목ID"], task_id)
    print("사전평가 설정 검사 통과")

def get_prework_questions(task_id, answers=None):
    """시점과 선행응답을 확인하여 적용/제외/대기 상태를 반환합니다."""
    task_id = str(task_id).strip()
    answers = normalize_answers({} if answers is None else answers)
    task_questions = get_questions_for_task(task_id)
    unexpected = set(answers) - set(task_questions["항목ID"])
    if unexpected:
        raise ValueError(f"작업 외 질문: {sorted(unexpected)}")

    rows = []
    for _, question in task_questions.iterrows():
        row = question.to_dict()
        policy = get_prework_policy(question["항목ID"], task_id)
        row["원래확인항목"] = question["확인항목"]
        row["확인항목"] = policy["사전확인항목"] or question["확인항목"]
        row["평가시점"] = "작업시작전"
        row["사전안내"] = policy["사전안내"]
        row["적용상태"] = "적용"
        row["제외사유"] = ""
        if not read_flag(policy["사전평가사용"], "사전평가사용"):
            row["적용상태"] = "시점제외"
            row["제외사유"] = policy["사전안내"]
        elif policy["제외조건_JSON"]:
            condition = json.loads(policy["제외조건_JSON"])
            parents = task_questions.loc[
                task_questions["점수규칙ID"] == condition["rule_id"]
            ]
            if parents.empty:
                # 선행질문 연결이 없는 경우 안전한 것으로 간주하지 않습니다.
                row["적용상태"] = "선행응답필요"
                row["제외사유"] = "해당 작업에 선행질문이 연결되지 않음"
            else:
                values = []
                for _, parent in parents.iterrows():
                    parent_id = parent["항목ID"]
                    parent_policy = get_prework_policy(parent_id, task_id)
                    if not read_flag(parent_policy["사전평가사용"], "사전평가사용"):
                        raise ValueError("사전평가 제외조건이 시점제외 질문에 의존합니다.")
                    value = answers.get(parent_id)
                    if not is_missing(value):
                        # 허용값이 아닌 응답 때문에 하위 질문이 제외되지 않도록 검증
                        score_answer(parent_id, value)
                        values.append(str(value).strip())
                if len(values) != len(parents):
                    row["적용상태"] = "선행응답필요"
                    row["제외사유"] = "먼저 응답할 질문: " + ";".join(parents["항목ID"])
                elif all(value == condition["value"] for value in values):
                    row["적용상태"] = "조건제외"
                    row["제외사유"] = "선행질문의 응답이 CSV 제외조건에 해당"
        rows.append(row)
    columns = list(task_questions.columns) + [
        "원래확인항목", "평가시점", "사전안내", "적용상태", "제외사유",
    ]
    return pd.DataFrame(rows, columns=columns)

def check_required_answers(task_id, answers):
    answers = normalize_answers(answers)
    table = get_prework_questions(task_id, answers)
    applicable = table["적용상태"].eq("적용")
    required = table["필수여부"].map(lambda value: read_flag(value, "필수여부"))
    missing = table.loc[
        applicable & required & table["항목ID"].map(
            lambda question_id: is_missing(answers.get(question_id))
        ), ["항목ID", "확인항목"]
    ].reset_index(drop=True)
    pending = table.loc[table["적용상태"] == "선행응답필요", "항목ID"].tolist()
    summary = {
        "작업ID": str(task_id).strip(),
        "평가시점": "작업시작전",
        "필수질문수": int((applicable & required).sum()),
        "누락질문수": len(missing),
        "선행응답대기ID": pending,
        "제외질문ID": table.loc[
            table["적용상태"].isin(["시점제외", "조건제외"]), "항목ID"
        ].tolist(),
        "필수응답입력완료": missing.empty and not pending,
    }
    return summary, missing


def select_representatives(detail):
    candidates = detail.loc[detail["점수합산대상"]]
    return (
        candidates.sort_values(["가중점수", "항목ID"], ascending=[False, True])
        .drop_duplicates("중복집계그룹", keep="first")
    )

def get_basic_grade(risk_index):
    if risk_index is None:
        return "평가불가"
    number = to_number(risk_index, "위험지수")
    rules = get_rules("S045")
    matched = rules.loc[
        rules.apply(lambda rule: matches_numeric_rule(number, rule), axis=1)
    ]
    rule = get_single_row(matched, f"위험지수 {number}의 기본등급")
    return rule["입력값·조건"]

def calculate_task_risk(task_id, answers):
    """사전 평가 대상만 합산하고 제외 응답은 별도 로그로 보존합니다."""
    answers = normalize_answers(answers)
    check, missing = check_required_answers(task_id, answers)
    if not check["필수응답입력완료"]:
        raise ValueError(
            f"필수응답 누락: {missing['항목ID'].tolist()} / "
            f"선행응답 대기: {check['선행응답대기ID']}"
        )
    table = get_prework_questions(task_id, answers)
    if table.empty:
        raise ValueError("해당 작업에 연결된 질문이 없습니다.")
    results = []
    for _, question in table.iterrows():
        question_id = question["항목ID"]
        answer = answers.get(question_id)
        active_answer = question["적용상태"] == "적용" and not is_missing(answer)
        if active_answer:
            result = score_answer(question_id, answer)
        else:
            # 제외나 미응답을 '안전한 0점 응답'으로 저장하지 않습니다.
            result = {
                "항목ID": question_id, "확인항목": question["확인항목"],
                "사용자응답": answer, "단위": question["단위"],
                "점수규칙ID": question["점수규칙ID"], "조건순번": "",
                "점수": None, "가중치": None, "가중점수": 0.0,
                "최대가중점수": 0.0, "점수합산대상": False,
                "중복집계그룹": question["중복집계그룹"],
                "그룹집계방식": question["그룹집계방식"],
                "관련기준ID": question["연결기준ID"],
                "적용조건": "", "확정상태": "점수미산정",
            }
        result["확인항목"] = question["확인항목"]
        result["평가시점"] = "작업시작전"
        result["적용상태"] = question["적용상태"]
        result["제외사유"] = question["제외사유"]
        result["사전안내"] = question["사전안내"]
        result["응답상태"] = "미응답" if is_missing(answer) else "입력"
        results.append(result)
    detail = pd.DataFrame(results)
    representatives = select_representatives(detail)
    detail["집계대표여부"] = detail.index.isin(representatives.index)
    total = float(representatives["가중점수"].sum())
    maximum = float(representatives["최대가중점수"].sum())
    risk_index = total / maximum * 100 if maximum > 0 else None
    summary = {
        "작업ID": str(task_id).strip(), "평가시점": "작업시작전",
        "적용질문수": int(table["적용상태"].eq("적용").sum()),
        "제외질문수": len(check["제외질문ID"]),
        "입력질문수": sum(not is_missing(value) for value in answers.values()),
        "집계항목수": len(representatives),
        "총가중위험점수": total, "최대가중점수": maximum,
        "위험지수": risk_index,
        "기본위험등급(잠정)": get_basic_grade(risk_index),
        "강제경고": "미검증",
        "최종판정": "보류: 법령·잔여 적용조건 미검증",
        "계산상태": "사전 시험 계산" if maximum > 0 else "계산 불가: 합산 대상 없음",
    }
    return summary, detail


def get_top_risks(detail, count=3):
    if isinstance(count, bool) or not isinstance(count, int) or count < 1:
        raise ValueError("표시 개수는 1 이상의 정수여야 합니다.")
    positive = detail.loc[detail["집계대표여부"] & (detail["가중점수"] > 0)]
    return (
        positive.sort_values(["가중점수", "항목ID"], ascending=[False, True])
        .head(count)[["항목ID", "확인항목", "사용자응답", "가중점수", "관련기준ID"]]
        .reset_index(drop=True)
    )

validate_prework_policies()


사전평가 설정 검사 통과


## 5. H05 사전평가 예시

가상 응답입니다. Q026(작업 후 잔불 확인)은 입력하지 않습니다. 적용 질문 8개, 집계 항목 7개, 분자 13.5, 분모 30, 위험지수 45.0, 잠정등급 주의가 예상됩니다. 이전 40.91과의 차이는 0점이던 사후 질문의 분모 3을 제외했기 때문입니다.


In [22]:
h05_answers = {
    "Q001": "있음",
    "Q002": 5,
    "Q003": "미실시",
    "Q004": "미배치",
    "Q018": "용접방화포 미사용",
    "Q019": "없음",
    "Q020": "없음",
    "Q025": "적정",
}
summary, detail = calculate_task_risk("H05", h05_answers)
display(pd.DataFrame([summary]))
display(detail)
display(get_top_risks(detail))

prework_question_table = get_prework_questions("H05", h05_answers)
display(prework_question_table[["항목ID", "확인항목", "적용상태", "제외사유"]])


NameError: name 'calculate_task_risk' is not defined

## 6. 경계값과 필수응답 확인

등급 구간은 그대로 CSV를 사용합니다. 두 응답만 입력하면 사후 Q026은 누락으로 세지 않으며, Q001에 의존하는 Q002는 선행응답 대기 상태로 구분됩니다.


In [ ]:
boundary_results = [
    {"위험지수": value, "기본등급": get_basic_grade(value)}
    for value in [0, 24.99, 25, 49.99, 50, 74.99, 75, 100]
]
display(pd.DataFrame(boundary_results))

partial = {"Q003": "미실시", "Q004": "배치"}
missing_summary, missing_questions = check_required_answers("H05", partial)
display(pd.DataFrame([missing_summary]))
display(missing_questions)


,위험지수,기본등급
0,0.00,낮음
1,24.99,낮음
2,25.00,주의
3,49.99,주의
4,50.00,높음
5,74.99,높음
6,75.00,매우높음
7,100.00,매우높음


,작업ID,평가시점,필수질문수,누락질문수,선행응답대기ID,제외질문ID,필수응답입력완료
0,H05,작업시작전,7,5,[Q002],[Q026],False


,항목ID,확인항목
0,Q001,주변 가연물 존재 여부
1,Q018,용접방화포 사용 시 성능인증 제품 여부
2,Q019,작업점 하부 가연물의 불티 착화 우려 여부
3,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부
4,Q025,초기소화설비 비치·접근 가능 여부


## 7. 가연물이 없을 때 거리 질문 제외

Q001=없음이고 Q002를 생략해도 계산할 수 있습니다. Q004의 해당없음 여부나 하부·반대면 착화조건을 자동으로 판단하는 기능은 아닙니다.


In [ ]:
no_combustibles = dict(h05_answers)
no_combustibles["Q001"] = "없음"
no_combustibles.pop("Q002")
no_combustibles_summary, no_combustibles_detail = calculate_task_risk("H05", no_combustibles)
display(pd.DataFrame([no_combustibles_summary]))
display(no_combustibles_detail[["항목ID", "사용자응답", "적용상태", "점수", "가중점수", "최대가중점수", "제외사유"]])


,작업ID,평가시점,적용질문수,제외질문수,입력질문수,집계항목수,총가중위험점수,최대가중점수,위험지수,기본위험등급(잠정),강제경고,최종판정,계산상태
0,H05,작업시작전,7,2,7,6,10.5,25.5,41.176471,주의,미검증,보류: 법령·잔여 적용조건 미검증,사전 시험 계산


,항목ID,사용자응답,적용상태,점수,가중점수,최대가중점수,제외사유
0,Q001,없음,적용,0.0,0.0,4.5,
1,Q002,NaN,조건제외,NaN,0.0,0.0,선행질문의 응답이 CSV 제외조건에 해당
2,Q003,미실시,적용,3.0,4.5,4.5,
3,Q004,미배치,적용,3.0,6.0,6.0,
4,Q018,용접방화포 미사용,적용,0.0,0.0,0.0,
5,Q019,없음,적용,0.0,0.0,3.0,
6,Q020,없음,적용,0.0,0.0,3.0,
7,Q025,적정,적용,0.0,0.0,4.5,
8,Q026,NaN,시점제외,NaN,0.0,0.0,작업 종료 후 잔불·불티 확인은 사후 점검 항목. 사전 점수와 필수응답에서 제외하며...


## 8. 관련 근거와 사고사례

현재 사전평가의 집계 대표 위험요인만 근거 조회에 사용합니다. 사고사례는 설명용이며 점수에 더하지 않습니다.


In [ ]:
def get_related_standards(detail, count=3):
    rows = []
    for _, risk in get_top_risks(detail, count).iterrows():
        for standard_id in dict.fromkeys(split_ids(risk["관련기준ID"])):
            standard = get_single_row(
                standards.loc[standards["기준ID"] == standard_id], f"기준 {standard_id}"
            )
            rows.append({
                "항목ID": risk["항목ID"], "위험요인": risk["확인항목"],
                "사용자응답": risk["사용자응답"], "가중점수": risk["가중점수"],
                "기준ID": standard_id, "적용조건": standard["적용조건"],
                "필수조치": standard["필수조치"], "출처URL": standard["출처URL"],
                "판정상태": "관련 근거 조회 - 적용·위반 여부 미검증",
            })
    return pd.DataFrame(rows, columns=[
        "항목ID", "위험요인", "사용자응답", "가중점수", "기준ID",
        "적용조건", "필수조치", "출처URL", "판정상태",
    ])

accidents = read_database("03_사고데이터.csv")

def get_related_accidents(task_id, verified_only=True):
    task_id = str(task_id).strip()
    get_single_row(tasks.loc[tasks["작업ID"] == task_id], f"작업 {task_id}")
    selected = accidents.loc[
        accidents["작업ID"].map(lambda value: task_id in split_ids(value))
    ].copy()
    if verified_only:
        selected = selected.loc[selected["검수상태"] == "검수완료"]
    return selected[[
        "사고ID", "작업ID", "사고일자", "지역", "사고유형",
        "사고경위_원문", "사고원인_원문", "출처URL", "검수상태"
    ]].reset_index(drop=True)

display(get_related_standards(detail))
display(get_related_accidents("H05"))


,항목ID,위험요인,사용자응답,가중점수,기준ID,적용조건,필수조치,출처URL,판정상태
0,Q004,화재감시자 배치 여부,미배치,6.0,R002,용접·용단 장소가 법정 화재감시자 배치조건에 해당,배치조건 해당 시 화재감시자를 지정·배치하고 대피유도 업무 수행에 필요한 장비 지급,https://www.law.go.kr/LSW/lsInfoP.do?lsiSeq=27...,관련 근거 조회 - 적용·위반 여부 미검증
1,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미실시,4.5,R001,가연성물질이 있는 장소에서 화재위험작업 실시,"작업절차 수립, 위험물 현황 파악, 인근 가연물 방호, 소화기구 비치, 불꽃·불티 ...",https://www.law.go.kr/LSW/lsInfoP.do?lsiSeq=27...,관련 근거 조회 - 적용·위반 여부 미검증
2,Q001,주변 가연물 존재 여부,있음,1.5,R001,가연성물질이 있는 장소에서 화재위험작업 실시,"작업절차 수립, 위험물 현황 파악, 인근 가연물 방호, 소화기구 비치, 불꽃·불티 ...",https://www.law.go.kr/LSW/lsInfoP.do?lsiSeq=27...,관련 근거 조회 - 적용·위반 여부 미검증


,사고ID,작업ID,사고일자,지역,사고유형,사고경위_원문,사고원인_원문,출처URL,검수상태
0,A001,H05,43551.0,경기 용인시 수지구 성복동,화재,철골구조물 부재설치 용접작업 중 발생한 불티가 비산하여 화재 발생,용접작업 중 불티가 주변 가연성 마감재에 착화,https://www.moel.go.kr/local/pohang/news/notic...,검수완료


## 다음 단계

1. 가상 예시가 45.0으로 계산되는지 확인합니다.
2. 09_사전평가설정.csv의 사전안내를 검토하고 실제 작업 준비상태로 응답해 봅니다.
3. 화재감시자·불티방지조치 등의 해당없음 선택 가능 조건과 법령 조건을 별도로 구조화합니다. 미검증을 경고 없음으로 처리하지 않습니다.

가열장비 점검 등은 사용 전 알 수 없는 상태를 임의로 정상으로 채우지 않습니다. 현재 단계는 준비상태 평가 시제품이며 실제 사고 확률을 뜻하지 않습니다.


In [10]:
import pandas as pd

# H05 용접·용단 작업의 추가 확인 질문
# 점수를 매기는 질문이 아니라 적용조건을 확인하는 질문입니다.
watcher_questions = pd.DataFrame([
    {
        "추가질문ID": "W001",
        "구분": "장소조건",
        "질문": (
            "작업반경 11m 이내 건물구조 자체 또는 내부"
            "(개구부 등으로 개방된 부분 포함)에 가연성물질이 있는가?"
        ),
    },
    {
        "추가질문ID": "W002",
        "구분": "장소조건",
        "질문": (
            "작업반경 11m 이내의 바닥 하부에 가연성물질이 "
            "11m 이상 떨어져 있지만 불꽃에 의해 쉽게 발화될 우려가 있는가?"
        ),
    },
    {
        "추가질문ID": "W003",
        "구분": "장소조건",
        "질문": (
            "금속 칸막이·벽·천장·지붕의 반대쪽 면에 "
            "가연성물질이 인접하여 열전도·열복사로 발화될 우려가 있는가?"
        ),
    },
    {
        "추가질문ID": "W004",
        "구분": "예외검토",
        "질문": (
            "같은 장소에서 상시·반복적으로 용접·용단하는 경우의 "
            "설비 구비 등 법령상 배치 예외를 적용하려는가?"
        ),
    },
])

watcher_questions["허용응답"] = "예 / 아니오 / 확인불가"

with pd.option_context("display.max_colwidth", 100):
    display(watcher_questions)

,추가질문ID,구분,질문,허용응답
0,W001,장소조건,작업반경 11m 이내 건물구조 자체 또는 내부(개구부 등으로 개방된 부분 포함)에 가연성물질이 있는가?,예 / 아니오 / 확인불가
1,W002,장소조건,작업반경 11m 이내의 바닥 하부에 가연성물질이 11m 이상 떨어져 있지만 불꽃에 의해 쉽게 발화될 우려가 있는가?,예 / 아니오 / 확인불가
2,W003,장소조건,금속 칸막이·벽·천장·지붕의 반대쪽 면에 가연성물질이 인접하여 열전도·열복사로 발화될 우려가 있는가?,예 / 아니오 / 확인불가
3,W004,예외검토,같은 장소에서 상시·반복적으로 용접·용단하는 경우의 설비 구비 등 법령상 배치 예외를 적용하려는가?,예 / 아니오 / 확인불가


In [11]:
def review_h05_watcher_conditions(answers):
    """H05 추가 응답을 검토합니다. 점수나 최종판정은 변경하지 않습니다."""
    if not isinstance(answers, dict):
        raise ValueError("응답은 {'W001': '예', ...} 형식이어야 합니다.")

    ids = watcher_questions["추가질문ID"].tolist()
    unknown_ids = set(answers) - set(ids)

    if unknown_ids:
        raise ValueError(f"알 수 없는 추가질문ID: {unknown_ids}")

    normalized = {}
    for question_id in ids:
        value = answers.get(question_id, "확인불가")

        if not isinstance(value, str):
            raise ValueError(f"{question_id}: 문자열 응답이 필요합니다.")

        value = value.strip()
        if value not in {"예", "아니오", "확인불가"}:
            raise ValueError(
                f"{question_id}: 예 / 아니오 / 확인불가 중 입력하세요."
            )
        normalized[question_id] = value

    place_ids = watcher_questions.loc[
        watcher_questions["구분"].eq("장소조건"), "추가질문ID"
    ].tolist()

    exception_id = watcher_questions.loc[
        watcher_questions["구분"].eq("예외검토"), "추가질문ID"
    ].item()

    matched = [qid for qid in place_ids if normalized[qid] == "예"]
    uncertain = [
        qid for qid in place_ids if normalized[qid] == "확인불가"
    ]
    exception = normalized[exception_id]

    if matched:
        place_status = "장소조건 해당 응답 있음"
    elif uncertain:
        place_status = "장소조건 확인 필요"
    else:
        place_status = "입력상 세 장소조건 모두 비해당"

    if exception == "예":
        review_status = "예외 근거 및 현장 조건 검토 필요"
    elif exception == "확인불가":
        review_status = "예외 적용 여부 확인 필요"
    elif matched:
        review_status = "배치 대상 검토: 장소조건 해당, 예외 적용 요청 없음"
    elif uncertain:
        review_status = "미확인 장소조건 추가 확인 필요"
    else:
        review_status = "적용 제외 후보: 현장 근거 확인 필요"

    summary = {
        "작업ID": "H05",
        "대상항목ID": "Q004",
        "연결기준ID": "R002",
        "장소조건검토": place_status,
        "해당조건ID": ", ".join(matched) or "-",
        "미확인조건ID": ", ".join(uncertain) or "-",
        "검토결과": review_status,
        "해당없음자동승인": False,
    }

    detail = watcher_questions.copy()
    detail["응답"] = detail["추가질문ID"].map(normalized)

    return summary, detail

In [12]:
watcher_answers = {
    "W001": "예",
    "W002": "아니오",
    "W003": "아니오",
    "W004": "아니오",
}

watcher_summary, watcher_detail = review_h05_watcher_conditions(
    watcher_answers
)

display(pd.DataFrame([watcher_summary]))

with pd.option_context("display.max_colwidth", 100):
    display(watcher_detail)

,작업ID,대상항목ID,연결기준ID,장소조건검토,해당조건ID,미확인조건ID,검토결과,해당없음자동승인
0,H05,Q004,R002,장소조건 해당 응답 있음,W001,-,"배치 대상 검토: 장소조건 해당, 예외 적용 요청 없음",False


,추가질문ID,구분,질문,허용응답,응답
0,W001,장소조건,작업반경 11m 이내 건물구조 자체 또는 내부(개구부 등으로 개방된 부분 포함)에 가연성물질이 있는가?,예 / 아니오 / 확인불가,예
1,W002,장소조건,작업반경 11m 이내의 바닥 하부에 가연성물질이 11m 이상 떨어져 있지만 불꽃에 의해 쉽게 발화될 우려가 있는가?,예 / 아니오 / 확인불가,아니오
2,W003,장소조건,금속 칸막이·벽·천장·지붕의 반대쪽 면에 가연성물질이 인접하여 열전도·열복사로 발화될 우려가 있는가?,예 / 아니오 / 확인불가,아니오
3,W004,예외검토,같은 장소에서 상시·반복적으로 용접·용단하는 경우의 설비 구비 등 법령상 배치 예외를 적용하려는가?,예 / 아니오 / 확인불가,아니오


In [21]:
def check_h05_watcher_response(watcher_answers, deployment_answer):
    """추가 현장 조건과 Q004 응답을 비교합니다."""
    allowed = {"배치", "미배치", "해당없음"}

    if not isinstance(deployment_answer, str):
        raise ValueError("Q004 응답은 문자열이어야 합니다.")

    deployment_answer = deployment_answer.strip()

    if deployment_answer not in allowed:
        raise ValueError("Q004는 배치 / 미배치 / 해당없음 중 입력하세요.")

    # 앞서 만든 함수로 추가 응답을 검증하고 결과를 다시 계산
    condition_summary, condition_detail = (
        review_h05_watcher_conditions(watcher_answers)
    )

    place_answers = condition_detail.loc[
        condition_detail["구분"].eq("장소조건"), "응답"
    ]

    exception_answer = condition_detail.loc[
        condition_detail["구분"].eq("예외검토"), "응답"
    ].item()

    has_condition = place_answers.eq("예").any()
    has_unknown = place_answers.eq("확인불가").any()

    if deployment_answer == "해당없음":
        status = "해당없음 근거 검토 필요"
        message = "추가 응답만으로 해당없음을 승인하지 않습니다."

    elif exception_answer != "아니오":
        status = "예외 적용 검토 필요"
        message = (
            "배치 예외의 적용 의사 또는 근거를 확인해야 합니다. "
            f"현재 Q004 응답은 '{deployment_answer}'입니다."
        )

    elif has_condition and deployment_answer == "미배치":
        status = "배치 상태 확인 필요"
        message = (
            "장소조건에 해당하는 응답이 있고 예외 적용 요청은 없지만, "
            "화재감시자가 미배치로 입력되어 있습니다."
        )

    elif has_condition:
        status = "배치 응답 확인"
        message = (
            "장소조건에 해당하는 응답과 배치 응답이 확인되었습니다. "
            "이 결과만으로 전체 안전조치가 완료된 것은 아닙니다."
        )

    elif has_unknown:
        status = "현장 조건 확인 필요"
        message = "확인불가로 남아 있는 장소조건을 확인하세요."

    else:
        status = "적용 여부 근거 확인 필요"
        message = (
            "입력상 세 장소조건은 모두 비해당입니다. "
            "현장 근거 확인 전에는 적용 제외를 확정하지 않습니다."
        )

    result = {
        **condition_summary,
        "Q004응답": deployment_answer,
        "응답비교결과": status,
        "확인메시지": message,
        "최종판정": "보류: 현장 근거 및 나머지 기준 미검증",
    }

    return result, condition_detail

In [20]:
# 실제 현장에서 확인한 배치 상태를 입력하세요.
# 허용값: "배치", "미배치", "해당없음"
# 아래 "미배치"는 테스트용 예시입니다.
deployment_answer = "미배치"

watcher_check, watcher_check_detail = check_h05_watcher_response(
    watcher_answers=watcher_answers,
    deployment_answer=deployment_answer,
)

with pd.option_context("display.max_colwidth", 120):
    display(
        pd.DataFrame(
            watcher_check.items(),
            columns=["항목", "결과"],
        )
    )

,항목,결과
0,작업ID,H05
1,대상항목ID,Q004
2,연결기준ID,R002
3,장소조건검토,장소조건 해당 응답 있음
4,해당조건ID,W001
5,미확인조건ID,-
6,검토결과,"배치 대상 검토: 장소조건 해당, 예외 적용 요청 없음"
7,해당없음자동승인,False
8,Q004응답,미배치
9,응답비교결과,배치 상태 확인 필요


In [23]:
test_cases = [
    (
        "조건 해당 + 미배치",
        {"W001": "예", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "미배치",
        "배치 상태 확인 필요",
    ),
    (
        "조건 해당 + 배치",
        {"W001": "예", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "배치",
        "배치 응답 확인",
    ),
    (
        "현장 조건 미확인",
        {"W001": "확인불가", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "미배치",
        "현장 조건 확인 필요",
    ),
    (
        "배치 예외 검토 요청",
        {"W001": "예", "W002": "아니오",
         "W003": "아니오", "W004": "예"},
        "미배치",
        "예외 적용 검토 필요",
    ),
    (
        "해당없음 자동 승인 방지",
        {"W001": "아니오", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "해당없음",
        "해당없음 근거 검토 필요",
    ),
    (
        "모든 장소조건 비해당",
        {"W001": "아니오", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "미배치",
        "적용 여부 근거 확인 필요",
    ),
]

test_results = []

for name, answers, deployment, expected in test_cases:
    result, _ = check_h05_watcher_response(answers, deployment)
    actual = result["응답비교결과"]

    test_results.append({
        "테스트": name,
        "예상결과": expected,
        "실제결과": actual,
        "통과": actual == expected,
    })

test_table = pd.DataFrame(test_results)
display(test_table)

assert test_table["통과"].all(), "실패한 테스트의 결과를 확인하세요."
print(f"총 {len(test_table)}개 테스트 통과")

,테스트,예상결과,실제결과,통과
0,조건 해당 + 미배치,배치 상태 확인 필요,배치 상태 확인 필요,True
1,조건 해당 + 배치,배치 응답 확인,배치 응답 확인,True
2,현장 조건 미확인,현장 조건 확인 필요,현장 조건 확인 필요,True
3,배치 예외 검토 요청,예외 적용 검토 필요,예외 적용 검토 필요,True
4,해당없음 자동 승인 방지,해당없음 근거 검토 필요,해당없음 근거 검토 필요,True
5,모든 장소조건 비해당,적용 여부 근거 확인 필요,적용 여부 근거 확인 필요,True


총 6개 테스트 통과
